# Fine-Tune Cellpose for Segmentation

This notebook provides a workflow for fine-tuning Cellpose models (e.g., CPSAM) using your corrected masks.

## Supported Training Modes

- **`secondary_obj`**: Fine-tune for secondary object segmentation (matches `segment_second_objs_ml`)
- **`cells`**: Fine-tune for cell segmentation (matches `segment_cellpose` with cells=True)
- **`nuclei`**: Fine-tune for nuclei-only segmentation (matches `segment_cellpose` with cells=False)

## Workflow
1. Select training mode and configure channel indices
2. Load training data (images + corrected masks)
3. Apply data augmentation (critical for small datasets)
4. Train/fine-tune the model
5. Evaluate performance
6. Save the trained model
7. Test on new images

## Prerequisites
- Corrected masks saved as `.npy` or `.tif` files (labeled masks where each object has a unique ID)
- Corresponding images (TIFF format, multi-channel)

## Key Feature: Preprocessing Consistency
The preprocessing applied during training now matches the preprocessing used in deployment functions:
- **secondary_obj**: Log scaling + max normalization (matches `segment_second_objs_ml`)
- **cells**: Log scaling + percentile normalization + RGB conversion (matches `prepare_cellpose`)
- **nuclei**: Percentile normalization (matches `segment_cellpose_nuclei_rgb`)

This ensures your fine-tuned models perform optimally when deployed.

## <font color='red'>SET PARAMETERS</font>

### Training Mode
- `TRAINING_MODE`: Preprocessing mode that must match your deployment target
  - `"secondary_obj"`: For segment_second_objs_ml (single channel, log scaling)
  - `"cells"`: For segment_cellpose with cells=True (3-channel RGB)
  - `"nuclei"`: For segment_cellpose with cells=False (DAPI only)

### Training Data Paths
- `IMAGE_DIR`: Directory containing training images (TIFF format)
- `MASK_DIR`: Directory containing corrected masks (NPY format)

### Channel Configuration
- For `secondary_obj` mode: Set `SECOND_OBJ_CHANNEL` (e.g., CDPK1 channel)
- For `cells`/`nuclei` modes: Set `DAPI_INDEX`, `CYTO_INDEX`, `HELPER_INDEX`

### Model Configuration
- `BASE_MODEL`: Base Cellpose model to fine-tune from ("cpsam", "cyto3", "cyto2", "nuclei")
- `MODEL_NAME`: Name for the fine-tuned model
- `MODEL_SAVE_DIR`: Directory to save the trained model

In [ ]:
from pathlib import Path

# Training_mode - must match your deployment target
TRAINING_MODE = None #"secondary_obj", "cells", or "nuclei"

# Training data paths
IMAGE_DIR = Path("training_data/images")
MASK_DIR = Path("training_data/masks")

# For cells/nuclei modes: channels matching segment_cellpose
DAPI_INDEX = 0      # Nuclear/DAPI channel
CYTO_INDEX = None      # Cytoplasmic channel (only needed for cells mode)
HELPER_INDEX = None  # Optional helper channel for CPSAM (can be None)
SECOND_OBJ_CHANNEL = None

# Model configuration
BASE_MODEL = None  # Options: "cpsam", "cyto3", "cyto2", "nuclei"
MODEL_NAME = None
MODEL_SAVE_DIR = Path("models")

# Create directories if they don't exist
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training mode: {TRAINING_MODE}")
if TRAINING_MODE == "secondary_obj":
    print(f"  Channel index: {SECOND_OBJ_CHANNEL}")
elif TRAINING_MODE == "cells":
    print(f"  DAPI index: {DAPI_INDEX}, Cyto index: {CYTO_INDEX}, Helper index: {HELPER_INDEX}")
elif TRAINING_MODE == "nuclei":
    print(f"  DAPI index: {DAPI_INDEX}")

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tifffile import imread
from glob import glob

from lib.shared.cellpose_training import (
    load_training_data,
    augment_training_data,
    prepare_cellpose_training,
    train_cellpose,
    load_trained_model,
    predict_masks,
    evaluate_segmentation,
    visualize_comparison,
    visualize_training_sample,
)

# For preprocessing new images when testing (uses same function as training/deployment)
from lib.shared.segment_cellpose import prepare_cellpose

## <font color='red'>SET PARAMETERS</font>

### Training Hyperparameters
- `N_EPOCHS`: Number of training epochs 
- `LEARNING_RATE`: Initial learning rate. Defaults to 1e-5.
- `BATCH_SIZE`: Training batch size
- `TEST_FRACTION`: Fraction of data to hold out for testing. Defaults to 0.1.

### Augmentation Settings
- `USE_AUGMENTATION`: Enable/disable data augmentation
- `USE_ROTATIONS`: Apply 90/180/270 degree rotations
- `USE_FLIPS`: Apply horizontal/vertical flips
- `USE_INTENSITY_SCALING`: Apply random intensity scaling

In [ ]:
# Training hyperparameters
N_EPOCHS = None
LEARNING_RATE = 1e-5
BATCH_SIZE = None
TEST_FRACTION = 0.1
WEIGHT_DECAY = 0.1
GPU = True

# Augmentation settings (recommended for small datasets)
USE_AUGMENTATION = True
USE_ROTATIONS = True
USE_FLIPS = True
USE_INTENSITY_SCALING = True

## 1. Load Training Data

Load paired images and masks. Images should be in TIFF format, masks can be NPY or TIFF format (labeled masks where each object has a unique ID).

**Expected naming convention:**
- Images: `tile_001.tif`, `tile_002.tif`, etc.
- Masks: `tile_001_mask.npy` or `tile_001_mask.tif`, etc.

Or you can manually specify the paths below.

In [ ]:
# Find all image and mask files
image_files = sorted(IMAGE_DIR.glob("*.tif")) + sorted(IMAGE_DIR.glob("*.tiff"))
mask_files = sorted(MASK_DIR.glob("*.npy")) + sorted(MASK_DIR.glob("*.tif")) + sorted(MASK_DIR.glob("*.tiff"))

print(f"Found {len(image_files)} images and {len(mask_files)} masks")

if len(image_files) == 0:
    print("\n" + "="*60)
    print("No training data found!")
    print("\nPlease add your training data:")
    print(f"  - Images (.tif): {IMAGE_DIR.absolute()}")
    print(f"  - Masks (.npy or .tif):  {MASK_DIR.absolute()}")
    print("="*60)

In [ ]:
# Option 1: Auto-pair files by name (assumes matching names)
# This cell pairs images and masks that share the same base identifier (e.g., A1_200)
import re

def extract_identifier(filename):
    """Extract well + tile identifier (e.g., 'A1_200') from filename."""
    # Match pattern like A1_200, B12_150, etc.
    match = re.search(r'([A-Z]\d+_\d+)', filename)
    return match.group(1) if match else None

def pair_images_masks(image_files, mask_files):
    """Pair images with masks based on identifier matching."""
    paired_images = []
    paired_masks = []
    
    # Build mask dict keyed by identifier
    mask_dict = {}
    for m in mask_files:
        identifier = extract_identifier(m.stem)
        if identifier:
            mask_dict[identifier] = m
    
    for img_path in image_files:
        identifier = extract_identifier(img_path.stem)
        if identifier and identifier in mask_dict:
            paired_images.append(img_path)
            paired_masks.append(mask_dict[identifier])
            print(f"  Paired: {img_path.name} <-> {mask_dict[identifier].name}")
    
    return paired_images, paired_masks

if len(image_files) > 0 and len(mask_files) > 0:
    print("Pairing images and masks...")
    image_paths, mask_paths = pair_images_masks(image_files, mask_files)
    print(f"\nSuccessfully paired {len(image_paths)} image-mask pairs")


In [6]:
# Option 2: Manually specify paths if auto-pairing doesn't work
# Uncomment and modify this cell if needed

# image_paths = [
#     Path("training_data/images/tile_001.tif"),
#     Path("training_data/images/tile_002.tif"),
# ]
# mask_paths = [
#     Path("training_data/masks/tile_001_mask.npy"),
#     Path("training_data/masks/tile_002_mask.npy"),
# ]

In [ ]:
# Load the training data with mode-specific preprocessing
if len(image_paths) > 0:
    if TRAINING_MODE == "secondary_obj":
        images, masks = load_training_data(
            image_paths,
            mask_paths,
            mode="secondary_obj",
            channel_index=SECOND_OBJ_CHANNEL,
        )
    elif TRAINING_MODE == "cells":
        images, masks = load_training_data(
            image_paths,
            mask_paths,
            mode="cells",
            dapi_index=DAPI_INDEX,
            cyto_index=CYTO_INDEX,
            helper_index=HELPER_INDEX,
        )
    elif TRAINING_MODE == "nuclei":
        images, masks = load_training_data(
            image_paths,
            mask_paths,
            mode="nuclei",
            dapi_index=DAPI_INDEX,
        )
    else:
        raise ValueError(f"Unknown TRAINING_MODE: {TRAINING_MODE}")
    
    print(f"\nImage shape: {images[0].shape}")
    print(f"Image dtype: {images[0].dtype}")
    print(f"Mask shape: {masks[0].shape}")
    print(f"Number of objects in first mask: {masks[0].max()}")

## 2. Visualize Training Data

Inspect a few training samples to verify masks are correctly loaded.

In [ ]:
# Visualize first few training samples
n_samples_to_show = min(3, len(images))

for i in range(n_samples_to_show):
    fig = visualize_training_sample(
        images[i], 
        masks[i], 
        title=f"Training Sample {i+1}"
    )
    plt.show()

## 3. Data Augmentation

For small datasets (5-20 images), augmentation is critical for successful training.

This will expand your dataset by applying:
- 90°, 180°, 270° rotations (4x multiplier)
- Horizontal and vertical flips (2x multiplier)
- Intensity scaling (optional, adds more variation)

**Expected expansion**: 10 images → ~80-160 training samples

In [ ]:
if USE_AUGMENTATION:
    print(f"Original dataset: {len(images)} samples")
    
    aug_images, aug_masks = augment_training_data(
        images,
        masks,
        rotations=USE_ROTATIONS,
        flips=USE_FLIPS,
        intensity_scaling=USE_INTENSITY_SCALING,
    )
    
    print(f"Augmented dataset: {len(aug_images)} samples")
else:
    aug_images = images
    aug_masks = masks
    print(f"Augmentation disabled. Using {len(aug_images)} original samples.")

In [ ]:
# Visualize some augmented samples
if USE_AUGMENTATION:
    print("Sample augmented images:")
    indices = [0, len(aug_images)//4, len(aug_images)//2, 3*len(aug_images)//4]
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, idx in enumerate(indices):
        axes[0, i].imshow(aug_images[idx], cmap='gray')
        axes[0, i].set_title(f'Image {idx}')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(aug_masks[idx], cmap='nipy_spectral')
        axes[1, i].set_title(f'Mask {idx}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

## 4. Prepare Training/Test Split

In [ ]:
# Split into training and test sets
train_images, train_masks, test_images, test_masks = prepare_cellpose_training(
    aug_images,
    aug_masks,
    test_fraction=TEST_FRACTION,
    seed=42
)

print(f"\nTraining set: {len(train_images)} samples")
print(f"Test set: {len(test_images)} samples")

## 5. Train the Model

This cell will fine-tune the Cellpose model. Training progress will be displayed.

**Note**: Training can take 30-60 minutes depending on dataset size and GPU availability.

In [ ]:
# Train the model
print(f"Starting training...")
print(f"  Base model: {BASE_MODEL}")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  GPU: {GPU}")
print()

model = train_cellpose(
    train_images=train_images,
    train_masks=train_masks,
    test_images=test_images,
    test_masks=test_masks,
    base_model=BASE_MODEL,
    n_epochs=N_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    save_path=MODEL_SAVE_DIR,
    model_name=MODEL_NAME,
    gpu=GPU,
)

## 6. Visualize Predictions

Compare predictions from the fine-tuned model against ground truth.

In [ ]:
# Get predictions on test set
pred_masks = predict_masks(model, test_images)

# Visualize comparisons
n_to_show = min(5, len(test_images))

for i in range(n_to_show):
    fig = visualize_comparison(
        test_images[i],
        pred_masks[i],
        test_masks[i],
        title=f"Test Image {i+1}"
    )
    plt.show()

## 7. Save Model Information

Save training configuration for reproducibility.

In [ ]:
import json
from datetime import datetime

# Save training configuration
config = {
    "model_name": MODEL_NAME,
    "base_model": BASE_MODEL,
    "training_date": datetime.now().isoformat(),
    "n_epochs": N_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "n_original_samples": len(images),
    "n_augmented_samples": len(aug_images),
    "n_train_samples": len(train_images),
    "n_test_samples": len(test_images),
    "augmentation": {
        "enabled": USE_AUGMENTATION,
        "rotations": USE_ROTATIONS,
        "flips": USE_FLIPS,
        "intensity_scaling": USE_INTENSITY_SCALING,
    },
    # "metrics": {
    #     "finetuned": {
    #         "mean_iou": finetuned_metrics["mean_iou"],
    #         "mean_precision": finetuned_metrics["mean_precision"],
    #         "mean_recall": finetuned_metrics["mean_recall"],
    #         "mean_f1": finetuned_metrics["mean_f1"],
    #     }
    # }
}

config_path = MODEL_SAVE_DIR / f"{MODEL_NAME}_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Training configuration saved to: {config_path}")
print(f"\nModel saved to: {MODEL_SAVE_DIR / MODEL_NAME}")

## 8. Evaluate Performance (Optional)

Compare the fine-tuned model against the base model on the test set. This will be computationally intensive and time-consuming. Our recommendation is trying the new model directly in the phenotyping notebook.

In [13]:
# # Evaluate the fine-tuned model
# print("Evaluating fine-tuned model on test set...")
# finetuned_metrics = evaluate_segmentation(
#     model,
#     test_images,
#     test_masks,
# )

# print(f"\nFine-tuned Model Performance:")
# print(f"  Mean IoU:       {finetuned_metrics['mean_iou']:.3f}")
# print(f"  Mean Precision: {finetuned_metrics['mean_precision']:.3f}")
# print(f"  Mean Recall:    {finetuned_metrics['mean_recall']:.3f}")
# print(f"  Mean F1:        {finetuned_metrics['mean_f1']:.3f}")

In [ ]:
# # Compare with base model (optional)
# from cellpose import models as cp_models

# print("Evaluating base model for comparison...")
# base_model = cp_models.CellposeModel(gpu=GPU, model_type=BASE_MODEL)

# base_metrics = evaluate_segmentation(
#     base_model,
#     test_images,
#     test_masks,
# )

# print(f"\nBase Model ({BASE_MODEL}) Performance:")
# print(f"  Mean IoU:       {base_metrics['mean_iou']:.3f}")
# print(f"  Mean Precision: {base_metrics['mean_precision']:.3f}")
# print(f"  Mean Recall:    {base_metrics['mean_recall']:.3f}")
# print(f"  Mean F1:        {base_metrics['mean_f1']:.3f}")

# print(f"\n{'='*50}")
# print("Improvement Summary:")
# print(f"  IoU:       {finetuned_metrics['mean_iou'] - base_metrics['mean_iou']:+.3f}")
# print(f"  Precision: {finetuned_metrics['mean_precision'] - base_metrics['mean_precision']:+.3f}")
# print(f"  Recall:    {finetuned_metrics['mean_recall'] - base_metrics['mean_recall']:+.3f}")
# print(f"  F1:        {finetuned_metrics['mean_f1'] - base_metrics['mean_f1']:+.3f}")

## Integration with Analysis Notebooks

### For Cell/Nuclei Segmentation

If you trained with `TRAINING_MODE="cells"` or `"nuclei"`, update the `CELLPOSE_MODEL` parameter in your preprocessing notebook:

```python
# Instead of:
SECOND_OBJ_CELLPOSE_MODEL = "cpsam"

# For cell segmentation:
CELLPOSE_MODEL = "models/cpsam_cells"  # Path to your fine-tuned cell model

# For nuclei-only segmentation:
CELLPOSE_MODEL = "models/cpsam_nuclei"  # Path to your fine-tuned nuclei model
```

### For Secondary Object Segmentation

If you trained with `TRAINING_MODE="secondary_object"`, update the `SECOND_OBJ_CELLPOSE_MODEL`

```python
# Use:
SECOND_OBJ_CELLPOSE_MODEL = "models/cpsam_secondary_obj"  # Path to your fine-tuned model
```